In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.insert(0, "..")

import random
from datasets import load_dataset
import pandas as pd
import numpy as np
from evidently.metric_preset import DataQualityPreset
from evidently.report import Report
from app_config import AppConfig

/home/dinhln1/Desktop/hcmut_master/is_assignment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load dataset

In [3]:
config = AppConfig()

In [4]:
metadata = load_dataset(
    config.dataset.hf_dataset_path,
    name=config.dataset.subset,
    split="full",
    trust_remote_code=True,
)

In [5]:
example = metadata[random.randint(0, len(metadata))]

example

{'main_category': 'AMAZON FASHION',
 'title': 'Outback Trading Co Woodbury Womens Jacket Bronze Oilskin XL',
 'average_rating': 2.0,
 'rating_number': 1,
 'features': [],
 'description': [],
 'price': 'None',
 'images': {'hi_res': ['https://m.media-amazon.com/images/I/61wunTSJkFL._AC_UL1024_.jpg'],
  'large': ['https://m.media-amazon.com/images/I/411EmGtFAfL._AC_.jpg'],
  'thumb': ['https://m.media-amazon.com/images/I/411EmGtFAfL._AC_SR38,50_.jpg'],
  'variant': ['MAIN']},
 'videos': {'title': [], 'url': [], 'user_id': []},
 'store': 'Outback Trading',
 'categories': [],
 'details': '{"Item Weight": "4 Pounds", "Item model number": "6186", "Date First Available": "February 13, 2017"}',
 'parent_asin': 'B06WRR39FF',
 'bought_together': None,
 'subtitle': None,
 'author': None}

In [6]:
raw_metadata_df = metadata.to_pandas()

In [7]:
def clean_empty(x):
    if isinstance(x, (list, tuple, np.ndarray)) and len(x) == 0:
        return None
    return x

raw_metadata_df["description"] = raw_metadata_df["description"].apply(clean_empty)
raw_metadata_df["features"] = raw_metadata_df["features"].apply(clean_empty)
raw_metadata_df["categories"] = raw_metadata_df["categories"].apply(clean_empty)

In [8]:
raw_metadata_df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,YUEDGE 5 Pairs Men's Moisture Control Cushione...,4.6,16,None,None,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GiveGift,None,"{""Package Dimensions"": ""10.31 x 8.5 x 1.73 inc...",B08BHN9PK5,NaN,NaN,NaN
1,AMAZON FASHION,DouBCQ Women's Palazzo Lounge Wide Leg Casual ...,4.1,7,"[Drawstring closure, Machine Wash]",None,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",DouBCQ,None,"{""Package Dimensions"": ""15 x 10.2 x 0.4 inches...",B08R39MRDW,NaN,NaN,NaN
2,AMAZON FASHION,Pastel by Vivienne Honey Vanilla Girls' Trapez...,4.3,11,"[Zipper closure, Hand Wash Only]",None,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Pastel by Vivienne,None,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...",B077KJHCJ4,NaN,NaN,NaN
3,AMAZON FASHION,Mento Streamtail,2.0,1,"[Thermoplastic Rubber sole, High Density Premi...",[Slip on the Women's Mento and you're ready to...,29.81,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Guy Harvey,None,"{""Package Dimensions"": ""11.22 x 4.72 x 4.33 in...",B0811M2JG9,NaN,NaN,NaN
4,AMAZON FASHION,RONNOX Women's 3-Pairs Bright Colored Calf Com...,4.3,3032,"[Pull On closure, Size Guide: ""S"" fits calf 10...",[Ronnox Calf Sleeves - Allowing Your Body to P...,17.99,{'hi_res': ['https://m.media-amazon.com/images...,{'title': ['HONEST Review: RONNOX Women's 3-Pa...,RONNOX,None,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...",B07SB2892S,NaN,NaN,NaN


## Basic Quality Report

In [9]:
len(raw_metadata_df)

826108

In [10]:
raw_metadata_df.describe()

,average_rating,rating_number
count,826108.000000,826108.000000
mean,3.910660,17.942204
std,0.982282,221.792730
min,1.000000,1.000000
25%,3.400000,2.000000
50%,4.000000,4.000000
75%,4.700000,10.000000
max,5.000000,46299.000000


In [11]:
is_null = raw_metadata_df.isnull().sum()
is_null

main_category           0
title                   0
average_rating          0
rating_number           0
features           363034
description        766819
price                   0
images                  0
videos                  0
store               26838
categories         826108
details                 0
parent_asin             0
bought_together    826108
subtitle           826108
author             826108
dtype: int64

In [12]:
len(raw_metadata_df) - is_null["description"]

np.int64(59289)

## Persit data to parquet

In [13]:
# Select only not-null description

raw_metadata_df_final = raw_metadata_df[raw_metadata_df["description"].notnull()]

raw_metadata_df_final.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
3,AMAZON FASHION,Mento Streamtail,2.0,1,"[Thermoplastic Rubber sole, High Density Premi...",[Slip on the Women's Mento and you're ready to...,29.81,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Guy Harvey,None,"{""Package Dimensions"": ""11.22 x 4.72 x 4.33 in...",B0811M2JG9,NaN,NaN,NaN
4,AMAZON FASHION,RONNOX Women's 3-Pairs Bright Colored Calf Com...,4.3,3032,"[Pull On closure, Size Guide: ""S"" fits calf 10...",[Ronnox Calf Sleeves - Allowing Your Body to P...,17.99,{'hi_res': ['https://m.media-amazon.com/images...,{'title': ['HONEST Review: RONNOX Women's 3-Pa...,RONNOX,None,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...",B07SB2892S,NaN,NaN,NaN
8,AMAZON FASHION,LYCKYY Women's Tie Dye Sweatshirt Crewneck Lon...,3.7,52,[Pull On closure],[Tie dye shirts for Women long sleeve crewneck...,9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",LYCKYY,None,"{""Department"": ""womens"", ""Date First Available...",B08FMLXY1Z,NaN,NaN,NaN
11,AMAZON FASHION,Sexyshine Women's Casual Fall Knit Long Sleeve...,3.6,7,"[Cotton Blend, Asian Size,Smaller than US Size...",[Sexyshine Women's Casual Fall Knit Long Sleev...,26.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Sexyshine,None,"{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B07G854X4J,NaN,NaN,NaN
37,AMAZON FASHION,Hugo Boss Mens Onyx Analog Casual Quartz Watch...,5.0,1,None,[Stainless steel case measuring 1.7 in (44 mm)...,395.0,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",HUGO by Hugo Boss,None,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...",B01GNVW8MC,NaN,NaN,NaN


In [14]:
len(raw_metadata_df_final)

59289

In [15]:
raw_metadata_df_final.to_parquet(config.dataset.local_path, index=False)